# 09. 전체 RAG 파이프라인 디버깅

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 전체 RAG 체인(질문→검색→프롬프트→생성→파싱)의 Trace를 단계별로 분석할 수 있어요
2. `client.list_runs(filter='gt(latency, "5s")')`로 느린 실행 구간을 프로그래밍 방식으로 찾을 수 있어요
3. 오류 Trace를 `filter='neq(error, null)'`로 필터링하여 원인을 추적할 수 있어요
4. RAG 진단 프레임워크를 사용하여 문제가 검색 단계인지 생성 단계인지 구분할 수 있어요
5. 실제 Trace에서 데이터를 추출하여 LangSmith 평가 데이터셋을 생성할 수 있어요

## 사전 지식

- 07, 08 노트북 완료 (VectorStore, Retriever 추적 이해)
- LCEL 체인 구성 방법 (04-Chain-and-LCEL-Tracing)
- `client.list_runs()` 기본 사용법 (08 노트북)

## RAG 파이프라인 전체 디버깅 전략

RAG 시스템에서 틀린 답이 나왔을 때, 문제의 원인은 크게 두 가지예요:
**검색 실패**(잘못된 문서 가져옴) 또는 **생성 실패**(좋은 문서가 있는데 잘못된 답 생성).

LangSmith는 이 두 단계를 명확히 분리해서 보여주기 때문에 원인 진단이 훨씬 쉬워요.

```mermaid
flowchart TD
    Q["❓ 사용자 질문"] --> CHAIN

    subgraph CHAIN["RAG Chain (Trace 1개)"]
        R["Retriever<br/>(Span 1)"] --> P["PromptTemplate<br/>(Span 2)"]
        P --> L["ChatOpenAI<br/>(Span 3)"]
        L --> O["StrOutputParser<br/>(Span 4)"]
    end

    O --> A["💬 최종 답변"]

    CHAIN --> LS["🔍 LangSmith Trace"]

    subgraph DIAG["진단 프레임워크"]
        D1["Span 1 출력 확인<br/>→ 관련 문서가 있는가?"] --> D2{"검색 결과<br/>품질은?"}
        D2 -->|좋음| D3["Span 3 입력 확인<br/>→ 프롬프트에 문서가 있는가?"]
        D2 -->|나쁨| FIX1["검색 설정 수정<br/>Retrieval Problem"]
        D3 --> D4{"프롬프트<br/>구성은?"}
        D4 -->|정상| D5["Span 3 출력 확인<br/>→ LLM이 무시했는가?"]
        D4 -->|비정상| FIX2["프롬프트 수정"]
        D5 -->|무시| FIX3["프롬프트 강화<br/>Generation Problem"]
    end

    LS --> DIAG

    classDef chain fill:#cce5ff,stroke:#007bff,color:#004085
    classDef diag fill:#d4edda,stroke:#28a745,color:#155724
    classDef fix fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef ls fill:#fff3cd,stroke:#ffc107,color:#856404

    class R,P,L,O chain
    class D1,D2,D3,D4,D5 diag
    class FIX1,FIX2,FIX3 fix
    class LS ls
```

### RAG 디버깅 3단계 체크리스트

| 단계 | 확인 위치 (LangSmith) | 질문 | 문제 유형 |
|------|----------------------|------|----------|
| 1. 검색 확인 | Retriever Span > Outputs | 반환된 문서가 질문과 관련 있는가? | Retrieval Problem |
| 2. 프롬프트 확인 | LLM Span > Inputs | 관련 문서가 프롬프트에 포함되었는가? | Prompt Assembly Problem |
| 3. 생성 확인 | LLM Span > Outputs | LLM이 문서를 무시하고 답했는가? | Generation Problem |

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 로드 및 LangSmith 설정
# ---------------------------------------------------
from dotenv import load_dotenv
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

load_dotenv()

# 이 노트북 전용 프로젝트 이름 설정
os.environ["LANGSMITH_PROJECT"] = "Ch09-RAG-Pipeline-Debug"

print(f"프로젝트: {os.environ.get('LANGSMITH_PROJECT')}")

프로젝트: Ch09-RAG-Pipeline-Debug


In [2]:
# ---------------------------------------------------
# RAG 파이프라인 공통 컴포넌트 초기화
# ---------------------------------------------------
# 이전 노트북과 동일한 VectorStore를 재구성해요
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.tracers.langchain import wait_for_all_tracers

# 문서 로딩 및 분할
loader = TextLoader("data/ai_concepts.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

# 임베딩 및 VectorStore
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents=split_docs, embedding=embeddings)

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"초기화 완료: {len(split_docs)}개 청크")

초기화 완료: 6개 청크


## 1. 전체 RAG 체인 구성 및 Trace

LCEL로 구성한 RAG 체인을 실행하면 LangSmith가 전체 파이프라인을 하나의 루트 Trace로 기록해요.
각 단계는 계층적 Span으로 표시돼요.

> 🎯 **강의 포인트**: RAG 체인 실행 후 LangSmith UI에서 Trace를 클릭해보세요. 왼쪽 트리뷰에서 Retriever → PromptTemplate → ChatOpenAI → StrOutputParser 순으로 계층 구조가 보여요. 각 Span을 클릭하면 해당 단계의 정확한 입력과 출력을 확인할 수 있어요. 이것이 RAG 디버깅의 핵심 도구예요.

In [3]:
# ---------------------------------------------------
# LCEL RAG 체인 구성
# ---------------------------------------------------
# 표준 RAG 체인: 검색 → 프롬프트 구성 → LLM 호출 → 파싱
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 검색기 설정
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# RAG 프롬프트 템플릿
rag_prompt = PromptTemplate.from_template("""
다음 문서를 참고하여 질문에 답해요. 문서에 없는 내용은 "문서에서 찾을 수 없어요"라고 답해요.

[참고 문서]
{context}

[질문]
{question}

[답변]
""")

# 문서를 텍스트로 변환하는 헬퍼 함수
def format_docs(docs):
    """검색된 Document 목록을 하나의 텍스트로 합쳐요"""
    return "\n\n".join(
        f"[문서 {i+1}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

# LCEL RAG 체인 조합
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료:")
print("  Retriever → PromptTemplate → ChatOpenAI → StrOutputParser")

RAG 체인 구성 완료:
  Retriever → PromptTemplate → ChatOpenAI → StrOutputParser


In [4]:
# ---------------------------------------------------
# 여러 질문으로 RAG 체인 실행
# ---------------------------------------------------
# 다양한 질문으로 Trace를 생성해요
from langchain_core.runnables import RunnableConfig

test_questions = [
    "RAG의 주요 장점은 무엇인가요?",
    "딥러닝과 머신러닝의 차이를 설명해요",
    "LangSmith는 어떤 기능을 제공하나요?",
    "벡터 데이터베이스 예시를 들어주세요",
]

print("RAG 체인 실행 중...")
answers = []

for i, question in enumerate(test_questions, 1):
    answer = rag_chain.invoke(
        question,
        config=RunnableConfig(
            run_name=f"rag_query_{i}",  # Trace 이름
            metadata={
                "question_id": i,
                "category": "test",
                "chain_version": "v1"
            }
        )
    )
    answers.append({"question": question, "answer": answer})
    print(f"  [{i}] 완료: {question[:40]}")

print(f"\n총 {len(answers)}개 질문 처리 완료")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서 각 Trace를 클릭하여 계층 구조를 확인해보세요!")

RAG 체인 실행 중...
  [1] 완료: RAG의 주요 장점은 무엇인가요?
  [2] 완료: 딥러닝과 머신러닝의 차이를 설명해요
  [3] 완료: LangSmith는 어떤 기능을 제공하나요?
  [4] 완료: 벡터 데이터베이스 예시를 들어주세요

총 4개 질문 처리 완료

👉 LangSmith UI에서 각 Trace를 클릭하여 계층 구조를 확인해보세요!


In [5]:
# ---------------------------------------------------
# 실행 결과 출력
# ---------------------------------------------------
print("RAG 체인 실행 결과:")
print("=" * 60)
for item in answers:
    print(f"\nQ: {item['question']}")
    print(f"A: {item['answer'][:200]}..." if len(item['answer']) > 200 else f"A: {item['answer']}")
    print("-" * 40)

RAG 체인 실행 결과:

Q: RAG의 주요 장점은 무엇인가요?
A: RAG의 주요 장점은 최신 정보를 활용할 수 있고, 할루시네이션(hallucination)을 줄일 수 있으며, 출처를 명확히 할 수 있다는 점입니다.
----------------------------------------

Q: 딥러닝과 머신러닝의 차이를 설명해요
A: 문서에서 찾을 수 없어요.
----------------------------------------

Q: LangSmith는 어떤 기능을 제공하나요?
A: LangSmith는 LLM 애플리케이션의 개발, 테스트, 모니터링을 위한 플랫폼으로, 모든 LLM 호출, 체인 실행, 에이전트 동작을 추적(Trace)하여 디버깅과 성능 최적화에 활용합니다. 또한, LangSmith를 통해 각 실행 단계의 입출력, 지연 시간, 토큰 사용량, 비용을 확인할 수 있습니다.
----------------------------------------

Q: 벡터 데이터베이스 예시를 들어주세요
A: FAISS, Chroma, Pinecone, Weaviate 등이 벡터 데이터베이스의 예시입니다.
----------------------------------------


## 2. 병목 식별: 느린 실행 찾기

RAG 파이프라인에서 응답이 느릴 때, 어떤 단계가 병목인지 LangSmith SDK로 찾을 수 있어요.

> 🔑 **핵심 개념**: `list_runs()` filter DSL에서 `gt(latency, "5s")`는 5초 이상 걸린 run만 필터링해요. `latency` 외에도 `total_tokens`, `error` 등을 필터링할 수 있어요. 단위는 문자열로 작성해요: `"5s"`, `"1m"`, `"100ms"`.

> 💡 **실무 팁**: 프로덕션에서는 Retriever 지연 시간 이상 징후를 모니터링하는 것이 중요해요. VectorDB 스케일링이 필요한 신호일 수 있어요. LangSmith 대시보드의 Latency 차트를 정기적으로 확인하세요.

In [6]:
# ---------------------------------------------------
# 의도적으로 느린 실행 생성 (병목 테스트용)
# ---------------------------------------------------
# 실제 환경에서 느린 실행을 시뮬레이션해요
import time
from langsmith import traceable

@traceable(name="slow_rag_chain", tags=["slow", "test"])
def simulate_slow_rag(question: str) -> str:
    """느린 RAG 체인 시뮬레이션 (병목 테스트용)"""
    # 검색 단계 지연 시뮬레이션
    time.sleep(1)  # 1초 대기 (외부 API 지연 등 시뮬레이션)
    docs = retriever.invoke(question)

    # 생성 단계
    context = format_docs(docs)
    answer = rag_chain.invoke(question)
    return answer

# 느린 실행 1회 실행
slow_result = simulate_slow_rag("임베딩 모델의 차원이란?")
print(f"느린 실행 완료: {slow_result[:100]}...")

wait_for_all_tracers()

느린 실행 완료: 문서에서 찾을 수 없어요....


In [ ]:
# ---------------------------------------------------
# 느린 run 필터링 (gt = greater than)
# ---------------------------------------------------
# 지정한 지연 시간 이상의 run을 찾아요
from langsmith import Client
from datetime import datetime, timedelta

client = Client()

# 1초 이상 걸린 run 조회 (테스트 환경이므로 짧게 설정)
slow_runs = list(client.list_runs(
    project_name="Ch09-RAG-Pipeline-Debug",
    start_time=datetime.now() - timedelta(hours=1),
    filter='gt(latency, "1s")',  # 1초 이상 걸린 run
    select=["id", "name", "start_time", "end_time", "run_type"],
))

print(f"1초 이상 걸린 run 수: {len(slow_runs)}")
print()

for run in slow_runs[:10]:
    if run.end_time and run.start_time:
        latency = (run.end_time - run.start_time).total_seconds()
        print(f"  [{run.run_type}] {run.name}: {latency:.2f}초")
    else:
        print(f"  [{run.run_type}] {run.name}: (시간 정보 없음)")

In [ ]:
# ---------------------------------------------------
# run_type별 평균 지연 시간 분석
# ---------------------------------------------------
# 어떤 단계(retriever, llm, chain)가 전체 시간을 차지하는지 파악해요
all_runs = list(client.list_runs(
    project_name="Ch09-RAG-Pipeline-Debug",
    start_time=datetime.now() - timedelta(hours=1),
    select=["id", "name", "run_type", "start_time", "end_time"],
))

# run_type별 지연 시간 집계
from collections import defaultdict

latency_by_type = defaultdict(list)
for run in all_runs:
    if run.end_time and run.start_time:
        latency = (run.end_time - run.start_time).total_seconds()
        latency_by_type[run.run_type].append(latency)

print("run_type별 평균 지연 시간:")
print("-" * 50)
for run_type, latencies in sorted(latency_by_type.items()):
    if latencies:
        avg = sum(latencies) / len(latencies)
        max_l = max(latencies)
        print(f"  {run_type:20s}: 평균 {avg:.2f}s, 최대 {max_l:.2f}s ({len(latencies)}개)")

if not latency_by_type:
    print("  (조회된 데이터가 없어요. 잠시 후 다시 시도해보세요.)")

## 3. 오류 추적

실패한 실행을 `neq(error, null)` 필터로 찾아 오류 원인을 분석해요.

> ⚠️ **자주 하는 실수**: 오류가 발생해도 LangSmith에 Trace가 기록되어요. 오류를 try/except로 무조건 잡아서 숨기면 LangSmith에서도 추적이 어려워지니, 되도록 Trace가 실패 상태로 기록되게 두는 것이 디버깅에 유리해요.

In [ ]:
# ---------------------------------------------------
# 의도적 오류 발생 (오류 추적 테스트)
# ---------------------------------------------------
# 잘못된 입력으로 오류를 발생시켜 LangSmith에 기록해요
@traceable(name="error_rag_test", tags=["error", "test"])
def rag_with_error(question: str) -> str:
    """오류 발생 시나리오 테스트"""
    if not question or len(question.strip()) == 0:
        raise ValueError("질문이 비어있어요. 유효한 질문을 입력해주세요.")
    return rag_chain.invoke(question)

# 정상 실행
try:
    result = rag_with_error("파인튜닝이란?")
    print(f"정상 실행: {result[:100]}...")
except Exception as e:
    print(f"오류: {e}")

# 오류 발생 (빈 질문)
try:
    result = rag_with_error("")
    print(f"결과: {result}")
except ValueError as e:
    print(f"예상된 오류 발생: {e}")
    print("  → LangSmith에 오류 Trace가 기록되었어요")

wait_for_all_tracers()

In [ ]:
# ---------------------------------------------------
# 오류 run 필터링
# ---------------------------------------------------
# error=True 파라미터로 오류가 발생한 run만 조회해요
error_runs = list(client.list_runs(
    project_name="Ch09-RAG-Pipeline-Debug",
    start_time=datetime.now() - timedelta(hours=1),
    error=True,  # 오류가 있는 run만
    select=["id", "name", "run_type", "error", "inputs"],
))

print(f"오류 발생 run 수: {len(error_runs)}")
print()

for run in error_runs[:5]:
    print(f"  [{run.run_type}] {run.name}")
    if run.error:
        print(f"  오류: {str(run.error)[:100]}")
    if run.inputs:
        print(f"  입력: {str(run.inputs)[:80]}")
    print()

if len(error_runs) == 0:
    print("  (오류 run이 없어요. 잠시 후 다시 시도해보세요.)")

## 4. RAG 진단 프레임워크

"답이 틀렸다"는 문제를 체계적으로 진단하는 방법이에요.
LangSmith의 Trace를 단계별로 검사하면 문제의 원인을 정확하게 찾을 수 있어요.

> 🔑 **핵심 개념**: RAG 진단의 핵심은 **각 단계의 입출력을 명확히 보는 것**이에요.
> - **검색 문제**: Retriever 출력을 봐서 관련 없는 문서가 나오면 → 검색 설정 개선 필요
> - **생성 문제**: Retriever 출력은 좋은데 LLM 출력이 나쁘면 → 프롬프트 개선 필요

In [ ]:
# ---------------------------------------------------
# 진단용 상세 RAG 함수
# ---------------------------------------------------
# 각 단계의 중간 결과를 확인할 수 있는 투명한 RAG 함수
@traceable(
    name="rag_diagnostic",
    tags=["diagnostic", "debug"],
)
def rag_with_diagnostics(question: str) -> dict:
    """진단용 RAG: 각 단계 결과를 반환해요"""

    # 단계 1: 검색 (Retriever)
    retrieved_docs = retriever.invoke(question)

    # 진단 포인트 1: 검색 결과가 질문과 관련 있는가?
    retrieval_quality = {
        "doc_count": len(retrieved_docs),
        "docs_preview": [doc.page_content[:100] for doc in retrieved_docs],
    }

    # 단계 2: 프롬프트 구성
    context = format_docs(retrieved_docs)
    formatted_prompt = rag_prompt.format(
        context=context,
        question=question
    )

    # 진단 포인트 2: 프롬프트에 관련 문서가 포함되었는가?
    prompt_quality = {
        "context_length": len(context),
        "prompt_length": len(formatted_prompt),
        "context_preview": context[:200],
    }

    # 단계 3: LLM 생성
    answer = llm.invoke(formatted_prompt)

    # 진단 포인트 3: LLM이 문서를 참고했는가?
    generation_quality = {
        "answer": answer.content,
        "answer_length": len(answer.content),
    }

    return {
        "question": question,
        "retrieval": retrieval_quality,
        "prompt": prompt_quality,
        "generation": generation_quality,
    }

In [ ]:
# ---------------------------------------------------
# 진단 실행 및 결과 분석
# ---------------------------------------------------
# 문서에 없는 내용을 질문하여 진단 프레임워크 테스트
diagnostic_questions = [
    "RAG의 장점은 무엇인가요?",           # 문서에 있는 정보
    "LangGraph는 무엇인가요?",             # 문서에 부분적으로 있는 정보
    "한국 AI 스타트업 현황은 어떤가요?",   # 문서에 없는 정보 (오프 토픽)
]

print("RAG 진단 실행:")
print("=" * 70)

for question in diagnostic_questions:
    result = rag_with_diagnostics(question)

    print(f"\n[질문] {result['question']}")
    print(f"  검색 문서 수: {result['retrieval']['doc_count']}개")
    print(f"  컨텍스트 길이: {result['prompt']['context_length']}자")
    print(f"  답변: {result['generation']['answer'][:150]}...")
    print()

    # 진단 결과 출력
    answer_lower = result['generation']['answer'].lower()
    if "문서에서 찾을 수 없어요" in result['generation']['answer']:
        print(f"  📌 진단: 검색 문제 또는 질문 범위 초과")
    else:
        print(f"  ✅ 진단: 정상 응답")

wait_for_all_tracers()

In [ ]:
# ---------------------------------------------------
# 두 RAG 버전 비교: 검색 설정 변경 효과
# ---------------------------------------------------
# 같은 질문에 대해 다른 검색 설정의 결과를 비교해요
comparison_question = "파인튜닝과 프롬프트 엔지니어링을 비교해요"

# 버전 A: k=2 (적은 검색 결과)
retriever_v1 = vectorstore.as_retriever(search_kwargs={"k": 2})
chain_v1 = (
    {"context": retriever_v1 | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

answer_v1 = chain_v1.invoke(
    comparison_question,
    config=RunnableConfig(
        run_name="rag_v1_k2",
        metadata={"version": "v1", "k": 2, "experiment": "k_comparison"}
    )
)

# 버전 B: k=5 (많은 검색 결과)
retriever_v2 = vectorstore.as_retriever(search_kwargs={"k": 5})
chain_v2 = (
    {"context": retriever_v2 | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

answer_v2 = chain_v2.invoke(
    comparison_question,
    config=RunnableConfig(
        run_name="rag_v2_k5",
        metadata={"version": "v2", "k": 5, "experiment": "k_comparison"}
    )
)

print(f"질문: {comparison_question}")
print(f"\n[버전 A: k=2]")
print(f"{answer_v1[:300]}")
print(f"\n[버전 B: k=5]")
print(f"{answer_v2[:300]}")

wait_for_all_tracers()
print("\n👉 LangSmith UI에서 두 Trace를 선택하여 'Compare' 버튼을 눌러보세요!")
print("   두 버전의 검색 결과와 최종 답변을 나란히 비교할 수 있어요.")

## 5. Trace에서 평가 데이터셋 생성

실제 서비스에서 쌓인 Trace를 활용하여 평가 데이터셋을 만들 수 있어요.
이렇게 만든 데이터셋은 RAG 성능을 정기적으로 평가하는 회귀 테스트에 활용돼요.

> 💡 **실무 팁**: 프로덕션 Trace에서 데이터셋을 만들 때는 오류 run을 제외하고(`error=False`), 사용자 피드백이 좋은 run만 선별하면 고품질 평가 데이터셋을 만들 수 있어요.

> ⚠️ **자주 하는 실수**: 같은 이름의 데이터셋을 중복 생성하려고 하면 오류가 발생해요. `try/except`로 이미 존재하는 경우를 처리하거나, 고유한 이름을 사용하세요.

In [ ]:
# ---------------------------------------------------
# 실행 기록 조회: 데이터셋 생성 재료
# ---------------------------------------------------
# 정상 실행된 루트 run만 조회해요 (오류 run 제외)
candidate_runs = list(client.list_runs(
    project_name="Ch09-RAG-Pipeline-Debug",
    start_time=datetime.now() - timedelta(hours=2),
    is_root=True,          # 루트 run만 (체인 전체 실행)
    error=False,           # 오류 없는 run만
    select=[
        "id",
        "name",
        "inputs",
        "outputs",
        "run_type",
    ],
))

print(f"데이터셋 후보 run 수: {len(candidate_runs)}")
print()

for run in candidate_runs[:5]:
    print(f"  [{run.run_type}] {run.name}")
    if run.inputs:
        input_preview = str(list(run.inputs.values())[0])[:60] if run.inputs else ""
        print(f"  입력: {input_preview}")
    if run.outputs:
        output_preview = str(list(run.outputs.values())[0])[:60] if run.outputs else ""
        print(f"  출력: {output_preview}")
    print()

In [ ]:
# ---------------------------------------------------
# LangSmith 데이터셋 생성
# ---------------------------------------------------
# 조회한 run들로 평가용 데이터셋을 만들어요
import uuid

# 고유한 데이터셋 이름 (중복 방지)
dataset_name = f"RAG 진단 테스트 데이터셋 ({datetime.now().strftime('%m%d_%H%M')})"

if candidate_runs:
    try:
        # 데이터셋 생성
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description="09 노트북 RAG 파이프라인 실행에서 생성된 테스트 데이터셋"
        )
        print(f"데이터셋 생성 완료: {dataset.name}")
        print(f"  데이터셋 ID: {dataset.id}")

        # 예시(examples) 생성
        examples = []
        for run in candidate_runs:
            if run.inputs and run.outputs:
                examples.append({
                    "inputs": run.inputs,
                    "outputs": run.outputs,
                })

        if examples:
            client.create_examples(
                dataset_id=dataset.id,
                examples=examples
            )
            print(f"  생성된 예시 수: {len(examples)}")
            print(f"\n👉 LangSmith UI > Datasets 메뉴에서 '{dataset_name}'을 확인해보세요!")
        else:
            print("  (예시 데이터가 없어요. inputs/outputs가 있는 run이 필요해요.)")

    except Exception as e:
        print(f"데이터셋 생성 중 오류: {e}")
        print("  이미 같은 이름의 데이터셋이 있거나 API 오류예요.")
else:
    print("  (후보 run이 없어요. RAG 체인을 먼저 실행해야 해요.)")
    print("  섹션 1의 rag_chain.invoke() 셀들을 먼저 실행해보세요.")

In [ ]:
# ============================================================
# TODO: 특정 조건으로 필터링하여 고품질 데이터셋을 만들어요
# 힌트: filter='eq(metadata.category, "test")'로 test 카테고리만 선별해요
# 예상 결과: metadata.category=test인 run만 데이터셋에 포함돼요
# ============================================================

# 특정 태그로 필터링한 run에서 데이터셋 만들기
filtered_runs = list(client.list_runs(
    project_name="Ch09-RAG-Pipeline-Debug",
    start_time=datetime.now() - timedelta(hours=2),
    is_root=True,
    error=False,
    # TODO: filter 파라미터 추가해보세요
    # filter='eq(metadata.category, "test")',
    select=["id", "name", "inputs", "outputs"],
))

print(f"필터링된 run 수: {len(filtered_runs)}")
print("(filter 파라미터를 추가하면 특정 조건의 run만 선별할 수 있어요)")

## 6. LangSmith 대시보드 활용 가이드

이 섹션은 코드 없이 LangSmith UI를 활용하는 방법을 안내해요.

### 대시보드에서 확인할 수 있는 것

| 섹션 | 내용 | 활용 방법 |
|------|------|----------|
| **Traces** | 전체 Trace 목록, 지연 시간, 오류율 | 시간대별 성능 추이 모니터링 |
| **LLM Calls** | 모델별 호출 횟수, 토큰 사용량 | 비용 추정 및 이상 감지 |
| **Cost & Tokens** | 일별/시간별 비용 차트 | 예산 관리, 급격한 비용 증가 감지 |
| **Run Types** | retriever/chain/llm 비율 | 파이프라인 구성 파악 |
| **Feedback Scores** | 사용자 피드백 점수 분포 | 품질 추이 모니터링 |

### Trace 비교 방법

> 🎯 **강의 포인트**: LangSmith에서 두 Trace를 비교하는 방법이에요:
> 1. Trace 목록에서 비교할 Trace 체크박스 2개 선택
> 2. 우측 상단 'Compare' 버튼 클릭
> 3. 왼쪽/오른쪽에 두 Trace가 나란히 표시돼요
> 4. 각 Span의 입출력 차이를 직접 비교할 수 있어요
>
> 이 기능으로 k=2와 k=5 버전의 검색 결과 차이를 직접 비교해보세요!

In [ ]:
# ---------------------------------------------------
# 전체 진단 요약 출력
# ---------------------------------------------------
# 이 노트북에서 생성한 모든 Trace의 요약 정보
all_runs_summary = list(client.list_runs(
    project_name="Ch09-RAG-Pipeline-Debug",
    start_time=datetime.now() - timedelta(hours=2),
    is_root=True,
    select=["id", "name", "run_type", "start_time", "end_time", "error"],
))

total = len(all_runs_summary)
errors = sum(1 for r in all_runs_summary if r.error)
success = total - errors

latencies = []
for run in all_runs_summary:
    if run.end_time and run.start_time:
        latencies.append((run.end_time - run.start_time).total_seconds())

print("\n=" * 50)
print("RAG 파이프라인 디버깅 세션 요약")
print("=" * 50)
print(f"  총 실행 수: {total}")
print(f"  성공: {success}")
print(f"  오류: {errors}")
if latencies:
    print(f"  평균 지연 시간: {sum(latencies)/len(latencies):.2f}초")
    print(f"  최대 지연 시간: {max(latencies):.2f}초")
print()
print("👉 LangSmith 프로젝트 'Ch09-RAG-Pipeline-Debug'에서")
print("   위 통계를 시각적 대시보드로 확인할 수 있어요!")

wait_for_all_tracers()

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **전체 Trace 계층 구조**: LCEL RAG 체인의 각 단계(Retriever → PromptTemplate → LLM → Parser)가 계층적 Span으로 기록되어 단계별 입출력을 정확히 확인할 수 있어요
- **병목 식별**: `filter='gt(latency, "5s")'`로 느린 실행을 프로그래밍 방식으로 찾고, run_type별 평균 지연 시간을 분석할 수 있어요
- **오류 추적**: `filter='neq(error, null)'`로 실패한 실행만 필터링하여 오류 패턴을 분석할 수 있어요
- **RAG 진단 프레임워크**: 검색 결과 확인 → LLM 입력 확인 → LLM 출력 확인의 3단계로 문제가 검색인지 생성인지 구분해요
- **데이터셋 생성**: `is_root=True, error=False`로 정상 실행만 선별하여 회귀 테스트용 데이터셋을 만들 수 있어요
- **Trace 비교**: LangSmith UI의 Compare 기능으로 두 버전의 RAG 실행을 나란히 비교할 수 있어요

## 다음 노트북 예고

다음 `10-Playground.ipynb`에서는 **LangSmith Playground**를 배워요. Trace에서 직접 재실행하고 모델이나 프롬프트를 수정하여 결과가 어떻게 달라지는지 빠르게 실험하는 방법을 배워요.